## Сетап

Данный блокнот - продолжение базового

Здесь с помощью дополнительных данных о преимуществе сторон я пытаюсь увеличить `score` на тесте на метрике:

$$ \text{Gini} = 2 \cdot \text{ROC-AUC} - 1$$

Конечная цель — внедрить в пайплайн новую фичу, увилисить `gini`

## Импорты и настройка

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import joblib

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

sys.path.append(os.path.abspath('..'))

def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1.0

def sqrt_transformer(X):
    return np.sqrt(X)

def add_missing_flag(X):
    return X.isna().astype(float)

## Основная часть

### 1.1. Агрегации

В данных есть большой кусок про advantage — преимущество команды сил Света с точностью до минуты, по золоту и опыту, всё так же **в пределах 15 минут**. Лежат они в `dota_adv.csv`. Поработаю с ним

In [3]:
dota_adv = pd.read_csv('../data/raw/dota_adv.csv')
dota_adv.head()

,match_id,radiant_gold_adv,radiant_exp_adv
0,526846,[ 0 159 452 1904 2100 3290 3290 3290 3290 ...,[ 0 68 658 1397 1435 2118 2118 1923 1923 ...
1,511496,[ 0 -151 -141 12 -165 -151 -151 4 377 ...,[ 0 1 -136 243 -270 -8 -8 -169 -169 ...
2,90272,[],[]
3,153647,[],[]
4,694826,[],[]


In [4]:
def parse_adv(series):
    clean_series = series.str.strip('[]').str.replace(r'\s+', ' ', regex=True)
    clean_series = [np.fromstring(x, sep=' ', dtype=int).tolist() 
          for x in clean_series.astype(str)]
    result = np.array([xi + [np.nan] *(16 - len(xi)) for xi in clean_series])
    return result

gold_series = parse_adv(dota_adv['radiant_gold_adv'])
exp_series = parse_adv(dota_adv['radiant_exp_adv'])

Для начала возьмём простые агрегации: среднее, стандартное отклонение, доля минут, в которых radiant в преимуществе и последнее значение (на 15 минуте если есть)

In [5]:
mean_gold_agg = pd.Series(np.mean(gold_series, axis=1))
mean_exp_agg = pd.Series(np.mean(exp_series, axis=1))

std_gold_agg = pd.Series(np.std(gold_series, axis=1))
std_exp_agg = pd.Series(np.std(exp_series, axis=1))

frac_gold_agg = pd.Series([np.count_nonzero(i > 0) / np.size(i) for i in gold_series])
frac_exp_agg = pd.Series([np.count_nonzero(i > 0) / np.size(i) for i in exp_series])

last_gold_agg = pd.Series([i[-1] for i in gold_series])
last_exp_agg = pd.Series([i[-1] for i in exp_series])

dota_adv['mean_radiant_gold_adv'] = mean_gold_agg.astype(np.float64)
dota_adv['mean_radiant_exp_adv'] = mean_exp_agg.astype(np.float64)
dota_adv['std_radiant_gold_adv'] = std_gold_agg.astype(np.float64)
dota_adv['std_radiant_exp_adv'] = std_exp_agg.astype(np.float64)
dota_adv['frac_radiant_gold_adv'] = frac_gold_agg.astype(np.float64)
dota_adv['frac_radiant_exp_adv'] = frac_exp_agg.astype(np.float64)
dota_adv['last_radiant_gold_adv'] = last_gold_agg.astype(np.float64)
dota_adv['last_radiant_exp_adv'] = last_exp_agg.astype(np.float64)

dota_adv = dota_adv.drop(columns=['radiant_gold_adv', 'radiant_exp_adv'])
dota_adv = dota_adv.fillna(0)

Тут по логике из базового блокнота пишем `AdvantageEncoder`, который будет принимать матчи и возвращать наши агрегации по ним. Я его написал в `../src/transformers.py`. Протестим

In [6]:
matches_df_train = pd.read_csv('../data/raw/matches_df_train.csv', parse_dates=['date'])
matches_df_train = matches_df_train.sort_values(by=['date']).reset_index().drop(columns='index')
matches_df_test = pd.read_csv('../data/raw/matches_df_test.csv')

X_train, X_val, y_train, y_val = train_test_split(
    matches_df_train.drop(columns=['radiant_win']), matches_df_train['radiant_win'], shuffle=False, test_size=0.2
)

Возьмём наш предыдущий пайплайн и добавим новый трансформер в блок препроцессинга

In [7]:
from src.transformers import AdvantageEncoder

pipeline = joblib.load('../models/base_pipeline.pkl')
current_transformers = list(pipeline.named_steps['preprocessor'].transformers)
adv_pipeline = Pipeline([
    ('encoder', AdvantageEncoder(dota_adv)),
    ('scaler', StandardScaler())
])
new_step = ('advantages_prep', adv_pipeline, ['match_id'])
current_transformers.append(new_step)
pipeline.named_steps['preprocessor'].transformers = current_transformers

Пайплайн изменён, но дело в том, что в нём старая модель со старыми гиперпараметрами, а у нас уже новые признаки. **Оптимизируем** гиперпараметры для нового пайплайна с `optuna`

In [14]:
import optuna

def objective(trial):
    global pipeline, X_train, X_val, y_train, y_val

    params = {
        'C': trial.suggest_float('C', 1e-5, 1000, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l2']),
        'solver': 'lbfgs',
        'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
    }

    pipeline.set_params(classifier__C=params['C'], classifier__penalty=params['penalty'],
                        classifier__solver=params['solver'], classifier__max_iter=params['max_iter'])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict_proba(X_val)[:, 1]
    score = gini(y_val, y_pred)
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=25)

[I 2026-03-24 18:26:16,837] A new study created in memory with name: no-name-fc5b5c34-821c-4a77-9e89-a379b47fd23b


  0%|          | 0/25 [00:00<?, ?it/s]

C:\Users\dan\AppData\Local\Temp\ipykernel_1908\823691640.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[I 2026-03-24 18:26:21,343] Trial 0 finished with value: 0.35014899087059304 and parameters: {'C': 1.9967257618694373e-05, 'penalty': 'l2', 'max_iter': 1200}. Best is trial 0 with value: 0.35014899087059304.


C:\Users\dan\AppData\Local\Temp\ipykernel_1908\823691640.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


...

[I 2026-03-24 18:28:09,401] Trial 22 finished with value: 0.39390303386852143 and parameters: {'C': 0.01594772800646962, 'penalty': 'l2', 'max_iter': 1500}. Best is trial 19 with value: 0.3939078290672948.


C:\Users\dan\AppData\Local\Temp\ipykernel_1908\823691640.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[I 2026-03-24 18:28:14,169] Trial 23 finished with value: 0.39391052374887225 and parameters: {'C': 0.011389377672631101, 'penalty': 'l2', 'max_iter': 1400}. Best is trial 23 with value: 0.39391052374887225.


C:\Users\dan\AppData\Local\Temp\ipykernel_1908\823691640.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[I 2026-03-24 18:28:19,014] Trial 24 finished with value: 0.3939034999466471 and parameters: {'C': 0.01644854775514497, 'penalty': 'l2', 'max_iter': 1400}. Best is trial 23 with value: 0.39391052374887225.


Меняем модель на новую с оптимизированными параметрами

In [15]:
best_params = study.best_params
pipeline.steps[-1] = ('classifier', LogisticRegression(**best_params))
pipeline.fit(X_train, y_train)

c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section Example
  self[section] = content


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('heroes', ...), ('region', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [16]:
y_pred_train = pipeline.predict_proba(X_train)[:, 1]
y_pred_val = pipeline.predict_proba(X_val)[:, 1]

print(f'Gini на трейне: {gini(y_train, y_pred_train):.2f}\nGini на валидации: {gini(y_val, y_pred_val):.2f}')

Gini на трейне: 0.39
Gini на валидации: 0.39


Локальный предикт воодушевляет. Проверим на Kaggle

In [18]:
y_pred_test = pipeline.predict_proba(matches_df_test)[:, 1]
test_predictions = pd.DataFrame({
    'ID': matches_df_test['match_id'],
    'Value': y_pred_test
})
test_predictions.to_csv('../data/final/advanced_test_predictions.csv', index=False)

Предикт выдал значение 0.36983 на тесте в Kaggle. Предыдущая посылка была 0.26734

**Прирост огромный**

### 1.2. Тренд

Попробуем собрать агрегацию похитрее - она будет обозначать тренд, который есть в графиках преимущества, и если пословица верна, наша модель уловит эту зависимость. Подобная агрегация может помочь модели предсказать будущие значения преимуществ

Делать это я буду трансформером со следующим функционалом

1. Принимает на вход функцию колонку
2. Выделяет коэффициент наклона (`slope`, он же $`\alpha`$) при помощи МНК $(X^TX)^{-1}X^Ty$
3. Считает `r2` и `intercept` для одного advantage (тоже фичи)

Трансформер `TrendTransformer` я написал и сохранил там же - `../src/transformers.py`. Протестим

In [8]:
from src.transformers import TrendTransformer

current_transformers = list(pipeline.named_steps['preprocessor'].transformers)
trend_step = ('trend', TrendTransformer(gold_series, exp_series, dota_adv['match_id']), ['match_id'])
current_transformers.append(trend_step)
pipeline.named_steps['preprocessor'].transformers = current_transformers

In [12]:
pipeline.fit(X_train, y_train)

c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section Example
  self[section] = content


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('heroes', ...), ('region', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [14]:
import optuna 

def objective(trial):
    global pipeline, X_train, X_val, y_train, y_val

    params = {
        'C': trial.suggest_float('C', 1e-5, 1000, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l2']),
        'solver': 'lbfgs',
        'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
    }

    pipeline.set_params(classifier__C=params['C'], classifier__penalty=params['penalty'],
                        classifier__solver=params['solver'], classifier__max_iter=params['max_iter'])
    
    pipeline.fit(X_train.loc[:50000], y_train.loc[:50000])
    y_pred = pipeline.predict_proba(X_val)[:, 1]
    score = gini(y_val, y_pred)
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=25)

[I 2026-03-24 18:43:54,061] A new study created in memory with name: no-name-b8a52abf-6a98-4769-8033-e866fcf76a4f


  0%|          | 0/25 [00:00<?, ?it/s]

C:\Users\dan\AppData\Local\Temp\ipykernel_14608\1991426275.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\d

[I 2026-03-24 18:44:04,078] Trial 0 finished with value: 0.3934204092387683 and parameters: {'C': 12.865538818541035, 'penalty': 'l2', 'max_iter': 1500}. Best is trial 0 with value: 0.3934204092387683.


C:\Users\dan\AppData\Local\Temp\ipykernel_14608\1991426275.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\d

...

[I 2026-03-24 18:47:30,508] Trial 22 finished with value: 0.39355318501029135 and parameters: {'C': 458.99473941132015, 'penalty': 'l2', 'max_iter': 1800}. Best is trial 12 with value: 0.3936588527624225.


C:\Users\dan\AppData\Local\Temp\ipykernel_14608\1991426275.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[I 2026-03-24 18:47:32,897] Trial 23 finished with value: 0.29514272003413833 and parameters: {'C': 2.5913224782217556e-05, 'penalty': 'l2', 'max_iter': 2000}. Best is trial 12 with value: 0.3936588527624225.


C:\Users\dan\AppData\Local\Temp\ipykernel_14608\1991426275.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
c:\Users\dan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\d

[I 2026-03-24 18:47:42,804] Trial 24 finished with value: 0.39358833762579715 and parameters: {'C': 5.595034327266888, 'penalty': 'l2', 'max_iter': 1600}. Best is trial 12 with value: 0.3936588527624225.


In [15]:
y_pred_test = pipeline.predict_proba(matches_df_test)[:, 1]
test_predictions = pd.DataFrame({
    'ID': matches_df_test['match_id'],
    'Value': y_pred_test
})
test_predictions.to_csv('../data/final/advanced_test_predictions.csv', index=False)

**New score**: 0.37503

Скор чуть вырос, то есть фича работает

## Заключение

Сохраняем пайплайн

In [17]:
joblib.dump(pipeline, '../models/advanced_pipeline.pkl')

['../models/advanced_pipeline.pkl']